%md
## 📑 HAPS Payment MtD
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AQCbHQ)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
import boto3
import os

#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ["S3PREFIX"] 

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `p-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d')  

# SQL query to find the last working day of the previous month
last_working_day_df = spark.sql(f"""
SELECT CAST(LastWorkingDateOfMonth AS STRING) AS LastWorkingDateOfMonth
FROM (
    SELECT CalendarDate,
    CASE WHEN IsBusinessDay = 1 THEN CalendarDate
    ELSE PreviousBusinessDay
    END AS LastWorkingDateOfMonth
    FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE CalendarDate <= LAST_DAY(DATEADD(MONTH, -1, '{run_date}'))
) AS subquery
ORDER BY CalendarDate DESC
LIMIT 1
""").collect()[0]['LastWorkingDateOfMonth']

# Collect the result
# start_date = last_working_day_df.collect()[0]['LastWorkingDateOfMonth']
start_date = last_working_day_df

###### Schema of the table

In [0]:
table_name = 'HAPS_PaymentsMtD'

schema = (
"BusinessDate DATE,"
"PortfolioName string,"
"CDI string,"
"PartyName string,"
"AgreementKey string,"
"ExposureKey string,"
"AssetSegment string,"
"IsActiveFlag boolean,"
"SOMIsActiveFlag boolean,"
"ArrearsStatusENG string,"
"AmountEUR decimal(19,2),"
"SOMSourceBucket string,"
"SOMTotalLegalClaimEUR decimal(19,2),"
"SOMGBVEUR decimal(19,2),"
"SOMOnBalanceAmountEUR decimal(19,2),"
"SourceBucket string,"
"TotalLegalClaimEUR decimal(19,2),"
"GBVEUR decimal(19,2),"
"OnBalanceAmountEUR decimal(19,2),"
"ModificationType string,"
"ModificationOrigination string,"
"N3869Segment string,"
"SegmentationBusinessDate Date,"
"InsertTimeStamp TIMESTAMP"
)


In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date,  # type: ignore
)


##### Create Delta Tables with selected fields & Temporary Views for each delta table

##### assignemnt segmentation

In [0]:
AssignmentCutOffDate = spark.sql(
f'''
select 
CASE WHEN '{run_date}' <
(CASE WHEN DayName  = 'Saturday' 
THEN (SELECT NextBusinessDay FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarDate =  DATE(CONCAT(year('{run_date}'), '-', month('{run_date}'),'-16')))
ELSE (SELECT NextBusinessDay FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarDate =  DATE(CONCAT(year('{run_date}'), '-', month('{run_date}'),'-15')) ) END)
THEN '{run_date}'
ELSE 
(CASE WHEN DayName  = 'Saturday' 
THEN (SELECT NextBusinessDay FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarDate =  DATE(CONCAT(year('{run_date}'), '-', month('{run_date}'),'-16')))
ELSE (SELECT NextBusinessDay FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarDate =  DATE(CONCAT(year('{run_date}'), '-', month('{run_date}'),'-15')) ) END)
	END	as AssignmentCutOffDate
 FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarDate =  DATE(CONCAT(year('{run_date}'), '-', month('{run_date}'),'-14'))
'''
).collect()[0]['AssignmentCutOffDate']   

In [0]:
SegmentationDates = spark.sql(
 
f'''
 
SELECT B.PortfolioName
      ,B.SegmentationDate
      ,MAX(A._update_timestamp) AS _update_timestamp
FROM `{env_prefix}-ptools`.assignments.offer_collection  A
INNER JOIN (
             SELECT PortfolioName, MAX(BusinessDate) AS SegmentationDate
             FROM `{env_prefix}-ptools`.assignments.offer_collection
             WHERE BusinessDate <= CAST('{AssignmentCutOffDate}' AS DATE) --AND ExposureKeyWithPrefix IS NOT NULL
             AND PortfolioName is NOT null
             GROUP BY PortfolioName
           ) B
ON A.PortfolioName  = B.PortfolioName AND A.BusinessDate = B.SegmentationDate    
--WHERE A.ExposureKeyWithPrefix IS NOT NULL
GROUP BY B.PortfolioName
        ,B.SegmentationDate      

'''
) 
SegmentationDates.createOrReplaceTempView("SegmentationDates")
 

In [0]:
SegmentationSoM = spark.sql(
f'''
SELECT distinct N3869Segment, exposurekey, BusinessDate AS SegmentationBusinessDate

FROM `{env_prefix}-ptools`.assignments.offer_collection AS E
INNER JOIN SegmentationDates as SD ON E.BusinessDate = SD.SegmentationDate 
AND E.PortfolioName = SD.PortfolioName
and e._update_timestamp = sd._update_timestamp
WHERE E.RelationType = 'Primary Owner'
'''
)
SegmentationSoM.createOrReplaceTempView("SegmentationSoM")

##### haps_portfolios

In [0]:
haps_portfolios = spark.sql(
f'''
SELECT DISTINCT A.PortfolioName
FROM `{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios  A
INNER JOIN (SELECT BusinessDate, MAX(IngestionTimeStamp) IngestionTimeStamp, PortfolioName 
            FROM `{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios
            GROUP BY  BusinessDate, PortfolioName) B
ON A.BusinessDate = B.BusinessDate AND A.PortfolioName = B.PortfolioName AND A.IngestionTimeStamp = B.IngestionTimeStamp
WHERE A.Report = 'HAPS_PaymentsMtD'
AND A.IsActive = 1 
''' 
)
haps_portfolios.createOrReplaceTempView("haps_portfolios")

##### dexposure

In [0]:
dexposure_active = spark.sql(
f'''
SELECT 
    A.BusinessDate
    , A.ExposureKey
    , SUBSTR(A.ExposureKey,4,LENGTH(A.ExposureKey)) as `ExposureKey_NoPrefix`
    , a.DaysPastDueWithMateriality as DaysPastDueWithMateriality
    , `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.OutstandingTotalBalance, A.BusinessDate) as OutstandingTotalBalance
    , `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.GBV, A.BusinessDate) as GBV
    , `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.OnBalance, A.BusinessDate) as OnBalance
    , A.Currency
    , A.IsActiveFlag
    , A.Opendate
    , A.ProductKey
    , A.PortfolioName
    , A.CDI
    , A.PartyName
    , A.AgreementKey
    , A.AssetSegment
    , A.SourceSystem
    , A.ExposureStatus
    , A.IsActiveFlag
FROM `{env_prefix}-cdh`.derivative.dexposure as A
INNER JOIN haps_portfolios AS HP
ON HP.PortfolioName = A.PortfolioName
WHERE A.BusinessDate in ('{run_date}', '{start_date}') 
AND A.IsActiveFlag = true
'''
)
dexposure_active.createOrReplaceTempView("dexposure_active")

In [0]:

dexposure_inactive = spark.sql(
f'''
SELECT 
    {run_date!r} as BusinessDate
    , A.ExposureKey
    , SUBSTR(A.ExposureKey,4,LEN(A.ExposureKey)) as `ExposureKey_NoPrefix`
    , A.DaysPastDueWithMateriality
    , `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.OutstandingTotalBalance, A.BusinessDate) as OutstandingTotalBalance
    , `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.GBV, A.BusinessDate) as GBV
    , `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.OnBalance, A.BusinessDate) as OnBalance
    , A.Currency
    , A.IsActiveFlag
    , A.Opendate
    , A.ProductKey
    , A.PortfolioName
    , A.CDI
    , A.PartyName
    , A.AgreementKey
    , A.AssetSegment
    , A.SourceSystem
    , A.ExposureStatus 
    , A.IsActiveFlag
from (select e.*
      , rank() over (partition by e.ExposureKey order by e.BusinessDate desc) as Rank
      from `{env_prefix}-cdh`.derivative.dexposure e
      INNER JOIN haps_portfolios AS HP
      ON HP.PortfolioName = e.PortfolioName
      where e.SourceSystem = 'profits'
        and e.IsActiveFlag = false
        -- and e.BusinessDate > '{start_date}'
        and e.BusinessDate <= '{run_date}'
        ) AS A
left anti join dexposure_active e_pf_act on e_pf_act.ExposureKey = A.ExposureKey and e_pf_act.BusinessDate = '{run_date}'
where A.Rank = 1
'''
)
dexposure_inactive.createOrReplaceTempView("dexposure_inactive")

In [0]:
from pyspark.sql.functions import when, col

def get_bucket_from_dpd(dpd):
    return (when(dpd.isNull(), 0)
            .when(dpd <= 0, 0)
            .when((dpd >= 1) & (dpd <= 29), 1)
            .when((dpd >= 30) & (dpd <= 59), 2)
            .when((dpd >= 60) & (dpd <= 89), 3)
            .when((dpd >= 90) & (dpd <= 119), 4)
            .when((dpd >= 120) & (dpd <= 149), 5)
            .when((dpd >= 150) & (dpd <= 179), 6)
            .when((dpd >= 180) & (dpd <= 209), 7)
            .when((dpd >= 210) & (dpd <= 239), 8)
            .when((dpd >= 240) & (dpd <= 269), 9)
            .when((dpd >= 270) & (dpd <= 299), 10) 
            .when((dpd >= 300) & (dpd <= 329), 11)
            .when((dpd >= 330) & (dpd <= 359), 12)
            .when((dpd >= 360) & (dpd <= 720), 13)
            .when(dpd > 720, 14)
            .otherwise(None))

dexposure_active = dexposure_active.withColumn("SourceBucket", get_bucket_from_dpd(col("DaysPastDueWithMateriality")))
dexposure_active.createOrReplaceTempView("dexposure_active")

dexposure_inactive = dexposure_inactive.withColumn("SourceBucket", get_bucket_from_dpd(col("DaysPastDueWithMateriality")))
dexposure_inactive.createOrReplaceTempView("dexposure_inactive")

In [0]:
dexposure = spark.sql(
f'''
select *
from dexposure_active
where BusinessDate = {run_date!r}

union all

select *
from dexposure_inactive

'''
)
dexposure.createOrReplaceTempView("dexposure")

##### dexposure_som

In [0]:
dexposure_som = spark.sql(
f'''
select *
from dexposure_active
where BusinessDate = {start_date!r}

'''
)
dexposure_som.createOrReplaceTempView("dexposure_som")

##### product

In [0]:
product = spark.sql(
f'''
SELECT P.BusinessDate, P.ProductKey, P.ProductDescription
FROM `{env_prefix}-cdh`.primary.product as P
WHERE P.BusinessDate = {run_date!r}
'''
)
product.createOrReplaceTempView("product")

##### restructuring

In [0]:
restructuring = spark.sql(
f'''
SELECT distinct M.BusinessDate, M.NewExposureKey, M.RestructuringDate
FROM `{env_prefix}-cdh`.primary.restructuring as M
INNER JOIN haps_portfolios AS HP
ON HP.PortfolioName = M.PortfolioName
WHERE M.BusinessDate = {run_date!r}
'''
)
restructuring.createOrReplaceTempView("restructuring")

##### portfolio

In [0]:
portfolio = spark.sql(
f'''
SELECT P.BusinessDate, P.PortfolioName, P.AssignmentDate
FROM `{env_prefix}-cdh`.primary.portfolio P
INNER JOIN haps_portfolios AS HP
ON HP.PortfolioName = P.PortfolioName
WHERE P.BusinessDate = {run_date!r}
'''
)
portfolio.createOrReplaceTempView("portfolio")

##### paymentschedule

In [0]:
paymentschedule = spark.sql(
f'''
SELECT  ps.BusinessDate, ps.PortfolioName, ps.ExposureKey
FROM `{env_prefix}-cdh`.primary.paymentschedule as ps 
INNER JOIN haps_portfolios AS HP
ON HP.PortfolioName = PS.PortfolioName
WHERE ps.BusinessDate = {run_date!r}
'''
)
paymentschedule.createOrReplaceTempView("paymentschedule")

##### Modifications

In [0]:
Modifications = spark.sql(
f'''

            SELECT A.BusinessDate, A.ExposureKey, A.PortfolioName
            , CASE WHEN Prd.ProductDescription LIKE '%ΥΠΟ ΑΙΡΕΣΗ%' THEN 'Conditional Woff' ELSE 'Restructure' END `ModificationType`
            ,	CASE WHEN RIGHT(Prd.ProductKey,4) IN ('6078', '6079', '4015', '4016', '7362', '7363') THEN 'Cepal L3869' ELSE 'Cepal Restructure' END `ModificationOrigination`
            FROM dexposure as A
            INNER JOIN portfolio as P 
              ON p.PortfolioName = A.PortfolioName
            LEFT JOIN product Prd
              ON A.ProductKey = Prd.ProductKey
              AND A.BusinessDate = Prd.BusinessDate
            WHERE A.Opendate >= P.AssignmentDate
              AND A.BusinessDate = {run_date!r} AND A.IsActiveFlag = true

  UNION ALL

            SELECT  A.BusinessDate, A.ExposureKey, A.PortfolioName
            , CASE WHEN Prd.ProductDescription LIKE '%ΥΠΟ ΑΙΡΕΣΗ%' THEN 'Conditional Woff' ELSE 'Restructure' END `ModificationType`
            ,	CASE WHEN RIGHT(Prd.ProductKey,4) IN ('6078', '6079', '4015', '4016', '7362', '7363') THEN 'Migrated L3869' ELSE 'Migrated Restructure' END `ModificationOrigination`
            FROM dexposure as A
            INNER JOIN portfolio as P 
              ON p.PortfolioName = A.PortfolioName

            LEFT JOIN product Prd
              ON A.ProductKey = Prd.ProductKey
              AND A.BusinessDate = Prd.BusinessDate

            WHERE A.BusinessDate = {run_date!r} AND  A.Opendate < P.AssignmentDate AND A.IsActiveFlag = true
                  
            AND EXISTS (  SELECT DISTINCT ExposureKey 
                          FROM paymentschedule as ps 
                          WHERE ps.ExposureKey = A.ExposureKey
                          )

'''
)
Modifications.createOrReplaceTempView("Modifications")

##### Collections

In [0]:
Transactions_View = spark.sql(
f'''
with cte as (
SELECT distinct DCollectionKey, SourceExposureKey, actualdate, amount
FROM `{env_prefix}-cdh`.derivative.dcollection C 
INNER JOIN haps_portfolios AS HP
ON HP.PortfolioName = C.PortfolioName
WHERE C.ActualDate > DATEADD(MONTH, - 1, last_day({run_date!r}))
    AND C.ActualDate <= {run_date!r} 
    AND C.BusinessDate = (select max(BusinessDate) from `{env_prefix}-cdh`.derivative.dcollection ) 
    AND C.FinalReason = 'Matched')

select SourceExposureKey as exposurekey, cast(sum(Amount) as DECIMAL(19,2)) as AmountEUR, max(actualdate) as ActualDate
from cte 
group by exposurekey
'''
)
Transactions_View.createOrReplaceTempView("Transactions_View")

In [0]:
Collections = spark.sql(
f'''
SELECT 
      A.BusinessDate
    , A.ExposureKey
    ,	A.PortfolioName
    ,	A.CDI
    ,	A.PartyName
    ,	A.AgreementKey
    ,	A.AssetSegment
    ,	A.IsActiveFlag
    ,	A.ExposureStatus
    ,	A.SourceBucket
    , A.OutstandingTotalBalance
    ,	A.GBV
    ,	A.OnBalance		   
    ,	IFNULL(t1.AmountEUR,0) + IFNULL(t2.AmountEUR,0)  as AmountEUR
FROM dexposure as A
LEFT JOIN Transactions_View as T1 
  ON SUBSTR(A.ExposureKey,4,LEN(A.ExposureKey)) = T1.ExposureKey
LEFT JOIN Transactions_View as T2 
  ON A.ExposureKey = T2.ExposureKey
'''
)
Collections.createOrReplaceTempView("Collections")

##### PaymentsMtD

In [0]:
PaymentsMtD = spark.sql(
f'''
SELECT 
    {run_date!r} AS BusinessDate
  ,	C.PortfolioName
  ,	C.CDI
  ,	C.PartyName
  ,	C.AgreementKey
  ,	C.ExposureKey
  ,	C.AssetSegment
  ,	C.IsActiveFlag
  ,	A.IsActiveFlag AS SOMIsActiveFlag
  ,	C.ExposureStatus AS ArrearsStatusENG
  ,	C.AmountEUR
  ,	A.SourceBucket AS SOMSourceBucket
  ,	A.OutstandingTotalBalance AS SOMTotalLegalClaimEUR
  , A.GBV AS SOMGBVEUR
  , A.OnBalance AS SOMOnBalanceAmountEUR
  ,	C.SourceBucket
  ,	C.OutstandingTotalBalance AS TotalLegalClaimEUR
  ,	C.GBV AS GBVEUR
  ,	C.OnBalance AS OnBalanceAmountEUR
  ,	M.ModificationType
  ,	M.ModificationOrigination
  ,	SEG.N3869Segment
  ,SEG.SegmentationBusinessDate
FROM Collections as C 
LEFT JOIN dexposure_som as A
  ON A.ExposureKey = C.ExposureKey
LEFT JOIN Modifications as M 
  ON M.ExposureKey = C.ExposureKey
LEFT JOIN SegmentationSoM	SEG ON SEG.ExposureKey=  RIGHT(A.ExposureKey, LEN(A.ExposureKey) - 3)
WHERE C.AmountEUR <> 0
'''
)
PaymentsMtD.createOrReplaceTempView("PaymentsMtD")

In [0]:
final_df = spark.sql(
f'''
SELECT 
  CAST(A.BusinessDate AS DATE) AS BusinessDate
, CAST(PortfolioName AS string) AS PortfolioName
, CAST(CDI AS string) AS CDI
, CAST(PartyName AS string) AS PartyName
, CAST(AgreementKey AS string) AS AgreementKey
, CAST(ExposureKey AS string) AS ExposureKey
, CAST(AssetSegment AS string) AS AssetSegment
, CAST(IsActiveFlag AS boolean) AS IsActiveFlag
, CAST(SOMIsActiveFlag AS boolean) AS SOMIsActiveFlag
, CAST(ArrearsStatusENG AS string) AS ArrearsStatusENG
, CAST(AmountEUR AS decimal(19,2)) AS AmountEUR
, CAST(SOMSourceBucket AS string) AS SOMSourceBucket
, CAST(SOMTotalLegalClaimEUR AS decimal(19,2)) AS SOMTotalLegalClaimEUR
, CAST(SOMGBVEUR AS decimal(19,2)) AS SOMGBVEUR
, CAST(SOMOnBalanceAmountEUR AS decimal(19,2)) AS SOMOnBalanceAmountEUR
, CAST(SourceBucket AS string) AS SourceBucket
, CAST(TotalLegalClaimEUR AS decimal(19,2)) AS TotalLegalClaimEUR
, CAST(GBVEUR AS decimal(19,2)) AS GBVEUR
, CAST(OnBalanceAmountEUR AS decimal(19,2)) AS OnBalanceAmountEUR
, CAST(ModificationType AS string) AS ModificationType
, CAST(ModificationOrigination AS string) AS ModificationOrigination
, CAST(N3869Segment AS string) AS N3869Segment
, CAST(SegmentationBusinessDate AS Date) as SegmentationBusinessDate
, CAST(getdate() AS TIMESTAMP) AS InsertTimeStamp
FROM PaymentsMtD A
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(delivery_table_path)

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
HAPSPaymentsMtD = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
HAPSPaymentsMtD.createOrReplaceTempView("HAPSPaymentsMtD")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/HAPS/PaymentsMtD/" # 
file_name = "PaymentsMtD_" +run_date+ ".csv"
HAPSPaymentsMtD.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}") # 
rename_and_cleanup_files(output_path, file_name)